In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore", category=UserWarning)


In [2]:
RANDOM_STATE = 42
TARGET = "employed_status"
ID_COL = "anonymised_id"
N_SPLITS = 5

train = pd.read_csv("data/train.csv")
train = train.dropna(subset=["employed_status"])
test = pd.read_csv("data/test.csv")
groups_train = train[ID_COL]


Feature Engineering: Tenure, gated by prior employment and age x employed_lag and work_readiness_score x is_first_round

In [3]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["has_history"] = df["lag_round"].notna().astype(int)
    df["employed_lag_num"] = df["employed_lag"]  # 0/1/NaN
    df["employed_lag_x_recency"] = df["employed_lag_num"].fillna(0) * df["has_history"]

    # --- Tenure, gated by prior employment ---------------------------------
    df["tenure_lag_missing"] = df["tenure_lag"].isna().astype(int)
    df["tenure_lag"] = df["tenure_lag"].fillna(0)
    df["log_tenure_lag"] = np.log1p(df["tenure_lag"].clip(lower=0))
    df["log_tenure_lag_if_employed"] = df["log_tenure_lag"] * df["employed_lag_num"].fillna(0)

    df["is_first_round"] = (df["total_historical_rounds"] <= 1).astype(int)

    # --- Center age before squaring ---------------------------------
    df["age"] = df["age"].fillna(df["age"].median())
    age_mean = df["age"].mean()
    df["age_centered"] = df["age"] - age_mean
    df["age_sq"] = df["age_centered"] ** 2

    # --- age x employed_lag --------------------------------------------
    df["age_x_employed_lag"] = df["age"] * df["employed_lag_num"].fillna(0)

    # --- work_readiness_score x is_first_round --------------------------
    df["work_readiness_x_first_round"] = df["work_readiness_score"].fillna(
        df["work_readiness_score"].median()
    ) * df["is_first_round"]

    return df


In [4]:
def add_seasonality_features(df: pd.DataFrame, date_col: str = "survey_date") -> pd.DataFrame:
    """Add cyclical seasonality features: sin/cos of month."""
    df = df.copy()

    if date_col not in df.columns:
        df["month_sin"] = 0
        df["month_cos"] = 0
        df["month_sin_x_employed_lag"] = 0
        df["month_cos_x_employed_lag"] = 0
        return df

    month = pd.to_datetime(df[date_col]).dt.month

    df["month_sin"] = np.sin(2 * np.pi * month / 12)
    df["month_cos"] = np.cos(2 * np.pi * month / 12)

    df["month_sin_x_employed_lag"] = df["month_sin"] * df["employed_lag_num"].fillna(0)
    df["month_cos_x_employed_lag"] = df["month_cos"] * df["employed_lag_num"].fillna(0)

    return df


In [5]:
def make_interaction_categorical(df, col_a, col_b, new_col, min_count=None,
                                  train_ref=None, other_label="Other"):
    """Combine two categorical columns into one 'A||B' categorical.
    NaNs are stringified so 'Missing' combinations are preserved as their
    own category rather than dropped."""
    a = df[col_a].astype(str).fillna("Missing")
    b = df[col_b].astype(str).fillna("Missing")
    df[new_col] = a + "||" + b

    if min_count is not None:
        ref = train_ref if train_ref is not None else df
        counts = ref[new_col].value_counts()
        keep = set(counts[counts >= min_count].index)
        df[new_col] = df[new_col].where(df[new_col].isin(keep), other_label)
    return df


In [6]:
def add_frequency_encoding(train_df, test_df, col, new_col=None):
    new_col = new_col or f"{col}_freq"
    freq_map = train_df[col].value_counts(normalize=True)
    train_df[new_col] = train_df[col].map(freq_map).fillna(0)
    test_df[new_col] = test_df[col].map(freq_map).fillna(0)
    return train_df, test_df


def collapse_rare_categories(train_df, test_df, col, min_count=30, other_label="Other"):
    counts = train_df[col].value_counts()
    keep = set(counts[counts >= min_count].index)

    def _collapse(series):
        return series.where(series.isin(keep) | series.isna(), other_label)

    train_df[col] = _collapse(train_df[col])
    test_df[col] = _collapse(test_df[col])
    return train_df, test_df

def extract_month_from_date(df: pd.DataFrame, date_col: str = "survey_date") -> pd.Series:
    """Extract month from survey_date column."""
    if date_col not in df.columns:
        return pd.Series(np.nan, index=df.index)
    return pd.to_datetime(df[date_col]).dt.month


In [7]:
train = engineer_features(train)
test = engineer_features(test)

train = add_seasonality_features(train)
test = add_seasonality_features(test)


# --- combined interaction categoricals -------------------------------
train = make_interaction_categorical(train, "gender", "status_broad_lag",
                                      "gender_x_status_lag")
test = make_interaction_categorical(test, "gender", "status_broad_lag",
                                     "gender_x_status_lag")

train = make_interaction_categorical(train, "education_level", "status_broad_lag",
                                      "education_x_status_lag")
test = make_interaction_categorical(test, "education_level", "status_broad_lag",
                                     "education_x_status_lag")

# race x education_level: sparser combo, so collapse rare cells using
# TRAIN-only counts to avoid leakage.
train = make_interaction_categorical(train, "race", "education_level",
                                      "race_x_education", min_count=50,
                                      train_ref=train)
test = make_interaction_categorical(test, "race", "education_level",
                                     "race_x_education")

# map test's raw combos through the same keep-set as train (anything not
# seen with min_count in train becomes "Other")
_keep_race_edu = set(train["race_x_education"].unique()) - {"Other"}
test["race_x_education"] = test["race_x_education"].where(
    test["race_x_education"].isin(_keep_race_edu), "Other"
)


In [8]:
numeric_features = [
    "age",
    "age_sq",
    "employed_lag_x_recency",
    "log_tenure_lag_if_employed",
    "tenure_lag_missing",
    "total_historical_rounds",
    "has_history",
    "is_first_round",
    "work_readiness_score",
    "work_readiness_x_first_round",
    "age_x_employed_lag",
    "municipality_freq",
    "month_sin",
    "month_cos",
    "month_sin_x_employed_lag",
    "month_cos_x_employed_lag",
]

categorical_features = [
    "status_broad_lag",
    "gender",
    "race",
    "province",
    "education_level",
    "gender_x_status_lag",
    "education_x_status_lag",
]

numeric_features = [c for c in numeric_features if c in train.columns]
categorical_features = [c for c in categorical_features if c in train.columns]

y_train = train[TARGET].astype(int)


Pipeline (XGBoost, native categorical support via `enable_categorical=True`)

In [9]:
# Same fix as the LightGBM notebook: no sklearn Pipeline wrapper, so
# eval_set/early stopping is passed straight to the model's own .fit().
# XGBoost's sklearn API natively handles pandas 'category' dtype columns
# when enable_categorical=True and tree_method="hist" -- no need to
# one-hot or label-encode them ourselves.

XGB_PARAMS = dict(
    n_estimators=2000,
    learning_rate=0.02,
    max_depth=6,
    min_child_weight=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    tree_method="hist",
    enable_categorical=True,
    eval_metric="auc",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)


def to_category(df, cols, cat_maps=None):
    """Cast categorical columns to pandas 'category' dtype for XGBoost's
    native categorical splitting. When cat_maps is given (fit on train),
    val/test categories are aligned to it so unseen categories become NaN
    rather than a new category."""
    from pandas.api.types import CategoricalDtype
    df = df.copy()
    fitted = {}
    for c in cols:
        s = df[c].astype(str).fillna("Missing")
        if cat_maps is not None and c in cat_maps:
            df[c] = s.astype(CategoricalDtype(categories=cat_maps[c]))
        else:
            df[c] = s.astype("category")
            fitted[c] = df[c].cat.categories
    return df, fitted


Cross-Validation (walk-forward over the last 3 rounds, matching RF/Boosting notebooks)

In [10]:
rounds_sorted = sorted(train["current_round"].unique())
val_rounds = rounds_sorted[-3:]  # last 3 rounds as walk-forward cutoffs

fold_aucs = []
best_iters = []
subgroup_aucs = []

for cutoff in val_rounds:
    tr_df = train[train["current_round"] < cutoff].copy()
    val_df = train[train["current_round"] == cutoff].copy()

    if val_df.empty or tr_df.empty:
        continue

    tr_df, val_df = add_frequency_encoding(tr_df, val_df, "municipality")
    tr_df, val_df = collapse_rare_categories(tr_df, val_df, "education_level", min_count=100)

    tr_df, cat_maps = to_category(tr_df, categorical_features)
    val_df, _ = to_category(val_df, categorical_features, cat_maps)

    X_tr = tr_df[numeric_features + categorical_features]
    X_val = val_df[numeric_features + categorical_features]
    y_tr = tr_df[TARGET].astype(int)
    y_val = val_df[TARGET].astype(int)

    # class imbalance: use this fold's own ratio, mirroring class_weight="balanced"
    fold_scale_pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()

    model = xgb.XGBClassifier(
        **XGB_PARAMS,
        scale_pos_weight=fold_scale_pos_weight,
        early_stopping_rounds=50,
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    val_probs = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_probs)
    fold_aucs.append(auc)
    best_iters.append(model.best_iteration)

    has_hist = val_df["has_history"].values.astype(bool)
    sub = {}
    if has_hist.sum() > 20:
        sub["returning"] = roc_auc_score(y_val[has_hist], val_probs[has_hist])
    if (~has_hist).sum() > 20:
        sub["new_entrant"] = roc_auc_score(y_val[~has_hist], val_probs[~has_hist])
    subgroup_aucs.append(sub)

    print(f"Cutoff round {cutoff}: n_val={len(val_df)}, AUC={auc:.5f}, "
          f"best_iter={model.best_iteration}, subgroups={sub}")

print(f"\nWalk-forward mean AUC: {np.mean(fold_aucs):.5f} (+/- {np.std(fold_aucs):.5f})")
avg_best_iter = int(round(np.mean(best_iters)))
print(f"Average best_iteration across folds: {avg_best_iter}")


/tmp/ipykernel_626/1018515433.py:35: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  df[c] = s.astype(CategoricalDtype(categories=cat_maps[c]))


Cutoff round 6: n_val=4883, AUC=0.60527, best_iter=59, subgroups={'returning': 0.6937202785326086, 'new_entrant': 0.5675839913409666}


/tmp/ipykernel_626/1018515433.py:35: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  df[c] = s.astype(CategoricalDtype(categories=cat_maps[c]))


Cutoff round 7: n_val=3199, AUC=0.63195, best_iter=35, subgroups={'returning': 0.6348558356013095, 'new_entrant': 0.6333524765518256}


/tmp/ipykernel_626/1018515433.py:35: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  df[c] = s.astype(CategoricalDtype(categories=cat_maps[c]))


Cutoff round 8: n_val=2333, AUC=0.66496, best_iter=16, subgroups={'returning': 0.7442927972104173, 'new_entrant': 0.6470581009227458}

Walk-forward mean AUC: 0.63406 (+/- 0.02441)
Average best_iteration across folds: 37


Fit on the full training data and predict on the test set.\n\nAs with LightGBM, there's no held-out `eval_set` left for the full-data fit, so we reuse the average `best_iteration` from the walk-forward CV folds above as a fixed `n_estimators` instead of early stopping.

In [11]:
train, test = add_frequency_encoding(train, test, "municipality")
train, test = collapse_rare_categories(train, test, "education_level", min_count=100)

train_cat, cat_maps = to_category(train, categorical_features)
test_cat, _ = to_category(test, categorical_features, cat_maps)

X_train_final = train_cat[numeric_features + categorical_features]
X_test_final = test_cat[numeric_features + categorical_features]

full_scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

final_params = dict(XGB_PARAMS)
final_params["n_estimators"] = avg_best_iter

final_model = xgb.XGBClassifier(
    **final_params,
    scale_pos_weight=full_scale_pos_weight,
)
final_model.fit(X_train_final, y_train)

test_probs = final_model.predict_proba(X_test_final)[:, 1]

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    "employed_prob": test_probs,
})
import os
os.makedirs("Submissions", exist_ok=True)
submission.to_csv("Submissions/XGBoost_model.csv", index=False)
print("\nSaved XGBoost_model.csv")


/tmp/ipykernel_626/1018515433.py:35: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  df[c] = s.astype(CategoricalDtype(categories=cat_maps[c]))



Saved XGBoost_model.csv
